# RAG项目案例
以某东商品衣服为例，以衣服属性构建本地知识。可以自由更新本地知识，用户问题的答案也是基于本地知识生成的。
## 00. 这份Notebook是干什么的

将几份文本资料放进“知识库”，再让大模型回答问题时先查资料、再回答。

用处有 3 个：
1. 从 0 到 1 跑通这个 RAG 案例
2. 知道每个 Python 文件分别负责什么
3. 以后能把自己的文本资料替换进去，做成自己的知识库

## 01. 先看项目里有什么

这一步只是认目录。先知道项目里有哪些 `.py` 文件、有哪些示例 `.txt` 文件，后面就不会迷路。

In [1]:
import os
import sys
from pathlib import Path


def locate_project_root() -> Path:
    signatures = [("app_qa.py", "knowledge_base.py", "data/尺码推荐.txt")]
    seen: set[Path] = set()
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        resolved = path.resolve()
        if resolved not in seen and resolved.exists() and resolved.is_dir():
            seen.add(resolved)
            candidates.append(resolved)

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(base)
        try:
            child_dirs = [child for child in base.iterdir() if child.is_dir()]
        except PermissionError:
            child_dirs = []
        for child in child_dirs:
            add_candidate(child)
            try:
                grandchild_dirs = [grandchild for grandchild in child.iterdir() if grandchild.is_dir()]
            except PermissionError:
                grandchild_dirs = []
            for grandchild in grandchild_dirs:
                add_candidate(grandchild)

    for candidate in candidates:
        if all((candidate / rel).exists() for rel in signatures[0]):
            return candidate

    raise FileNotFoundError("没有找到 03_RAG项目案例 目录，请确认从仓库目录中打开 Notebook。")


project_root = locate_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

(project_root / 'chat_history').mkdir(exist_ok=True)
(project_root / 'chroma_db').mkdir(exist_ok=True)
print(f'当前工作目录: {project_root}')

python_files = sorted(p.name for p in project_root.glob('*.py'))
data_files = sorted(p.name for p in (project_root / 'data').glob('*.txt'))

print('项目中的 Python 文件:')
for name in python_files:
    print('-', name)

print('项目中的示例知识库文件:')
for name in data_files:
    print('-', name)


当前工作目录: d:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例
项目中的 Python 文件:
- app_file_uploader.py
- app_qa.py
- config_data.py
- file_history_store.py
- knowledge_base.py
- rag.py
- vector_stores.py
项目中的示例知识库文件:
- 尺码推荐.txt
- 洗涤养护.txt
- 颜色选择.txt


## 02. 安装运行依赖

如果是第一次跑这个案例，先执行这一格。它会安装 Streamlit、LangChain、Chroma、DashScope 等依赖。

如果之前已经装过，也可以跳过。

In [2]:
%pip install streamlit langchain-core langchain-community langchain-chroma langchain-text-splitters chromadb dashscope

  Using cached streamlit-1.56.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached narwhals-2.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
Using cached streamlit-1.56.0-py3-none-any.whl (9.1 MB)
Using cached altair-6.0.0-py3-none-any.whl (795 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached gitpython-3.1.46-py3-none-any.whl (208 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached pydeck-0.9.1-py2.py3-none-any.whl (6.9 MB)
Using cached smmap-5.0.3

## 03. 检查 API Key

这个项目要调用阿里云百炼，所以必须先有 `DASHSCOPE_API_KEY`。

没有它，就像手机没插 SIM 卡，后面的联网能力都不能用。

In [3]:
import os
import sys

# 看看 Python 版本
print('Python 版本:', sys.version.split()[0])

# 看看环境变量里有没有 API Key
api_key_exists = bool(os.getenv('DASHSCOPE_API_KEY'))
print('DASHSCOPE_API_KEY 是否已设置:', api_key_exists)

# 没有 Key 就先停下，因为后面肯定跑不通
if not api_key_exists:
    raise EnvironmentError('请先设置 DASHSCOPE_API_KEY，再继续运行后面的单元。')

Python 版本: 3.13.12
DASHSCOPE_API_KEY 是否已设置: True


## 04. 导入项目里的现成模块

这一节很重要。这里不是重写代码，而是直接复用项目里已经写好的模块。

可以先这样理解它们：
- `knowledge_base.py`：负责把文本放进知识库
- `vector_stores.py`：负责从知识库里查资料
- `rag.py`：负责把“查资料 + 问大模型”串起来
- `config_data.py`：负责放配置

In [4]:
import json
import shutil

# 导入项目中的配置和服务类
import config_data as config
from knowledge_base import KnowledgeBaseService
from vector_stores import VectorStoreService
from rag import RagService
from langchain_community.embeddings import DashScopeEmbeddings

# 打印关键配置，先有个整体印象
print('embedding 模型:', config.embedding_model_name)
print('chat 模型:', config.chat_model_name)
print('向量库目录:', config.persist_directory)
print('集合名:', config.collection_name)
print('检索返回条数 k:', config.similarity_threshold)

embedding 模型: text-embedding-v4
chat 模型: qwen3-max
向量库目录: ./chroma_db
集合名: rag
检索返回条数 k: 1


## 05. 可选：清空旧数据

如果想从头复现，最好先把旧的向量库、聊天记录、去重记录清空。

为了安全，这里默认不删除。只有你把 `RESET_DEMO_STATE` 改成 `True` 才会执行。

In [6]:
RESET_DEMO_STATE = False#True

# 这 3 个位置分别存放向量库、聊天记录、去重记录
targets = [Path(config.persist_directory), Path('chat_history'), Path(config.md5_path)]

if RESET_DEMO_STATE:
    for target in targets:
        if target.is_dir():
            shutil.rmtree(target)
            print(f'[已删除目录] {target}')
        elif target.exists():
            target.unlink()
            print(f'[已删除文件] {target}')
        else:
            print(f'[跳过] {target} 不存在')
else:
    print('当前是安全模式，没有删除任何文件。')
    print('如果你要从头复现，把 RESET_DEMO_STATE 改成 True 后再运行一次。')

当前是安全模式，没有删除任何文件。
如果你要从头复现，把 RESET_DEMO_STATE 改成 True 后再运行一次。


## 06. 看看示例文本里写了什么

在导入知识库之前，先看看原始文本内容。这样就会知道后面模型到底是根据什么资料来回答问题的。

In [8]:
data_dir = Path('data')
txt_files = sorted(data_dir.glob('*.txt'))

# 只预览每个文件的前 300 个字符
for txt_file in txt_files:
    print(f'===== {txt_file.name} =====')
    print(txt_file.read_text(encoding='utf-8')[:300])
    print('-' * 40)

===== 尺码推荐.txt =====
身高：155-165cm， 体重：75-95 斤，建议尺码S。
身高：160-170cm， 体重：90-115斤，建议尺码M。
身高：165-175cm， 体重：115-135斤，建议尺码L。
身高：170-178cm， 体重：130-150斤，建议尺码XL。
身高：175-182cm， 体重：145-165斤，建议尺码2XL。
身高：178-185cm， 体重：160-180斤，建议尺码3XL。
身高：180-190cm， 体重：180-210斤，建议尺码4XL。
身高：190cm+，体重：210斤+，建议尺码5XL。
----------------------------------------
===== 洗涤养护.txt =====
一、春季服装（纯棉、薄牛仔、针织棉、轻薄化纤）

1. 纯棉材质（春季衬衫、T恤、休闲裤）

洗涤：可机洗或手洗，水温≤30℃，中性洗涤剂；浅色与深色分开洗，首次洗加少许盐固色；机洗用洗衣袋+轻柔模式，避免摩擦起球。

养护：阴凉通风处阴干，避免暴晒褪色；收纳前完全干燥，折叠或宽肩悬挂；潮湿天放干燥剂防发霉。

2. 薄牛仔材质（春季牛仔裤、牛仔外套）

洗涤：水温≤30℃，中性洗涤剂；翻面清洗减少褪色，机洗选轻柔模式；避免频繁清洗，1-2周一次即可。

养护：翻面阴干，避免阳光直射；收纳时折叠平放或悬挂，宽肩衣架防止裤腰变形；裤兜内放防潮纸保持版型。

3. 针织棉材质（春季针织衫、薄开衫）
----------------------------------------
===== 颜色选择.txt =====
1.  肤色与服装颜色搭配原则
    冷白皮：适合冷色调和暖色调，亮色系（如宝蓝、正红、薄荷绿）更显白皙透亮；深色系（如黑色、深灰）可提升气场，避免过于苍白。
    黄皮/暖黄皮：优先选暖色调（如焦糖色、姜黄色、豆沙色），避免冷调荧光色（如荧光绿、冷粉），易显肤色暗沉；浅米色、燕麦色可柔和肤色，提升气色。
    黑皮：适合高饱和度亮色（如明黄、橙色、湖蓝），突出健康肤色；避免暗沉的土黄色、灰褐色，易显肤色暗沉无光。

2.  场合与服装颜色选择
    日常通勤：以基础色为主（黑白灰、米色、藏蓝），简约大气；可搭配低饱和度亮色（如雾霾蓝、浅紫）作为点缀，增加活力。
 

## 07. 把文本放进知识库

这一步就是把 `data` 目录下的文本喂给程序。程序会自动做 4 件事：
1. 读取文本
2. 必要时切分文本
3. 做向量化
4. 写入 Chroma 向量库

In [9]:
# 创建知识库服务对象
kb_service = KnowledgeBaseService()
import_results = []
# 逐个导入 txt 文件
for txt_file in txt_files:
    text = txt_file.read_text(encoding='utf-8')
    result = kb_service.upload_by_str(text, txt_file.name)
    import_results.append((txt_file.name, result))
    print(f'{txt_file.name}: {result}')

print('一共处理了多少个文件:', len(import_results))

尺码推荐.txt: [跳过]内容已经存在知识库中
洗涤养护.txt: [跳过]内容已经存在知识库中
颜色选择.txt: [跳过]内容已经存在知识库中
一共处理了多少个文件: 3


## 08. 看看本地有没有真的生成东西

刚开始在这里会疑惑：“我刚刚导入的东西到底存哪了？”

答案是：向量数据会写进 `chroma_db`，去重信息会写进 `md5.text`。下面直接看。

In [12]:
from pathlib import Path
import config_data as config

persist_path = Path(config.persist_directory)
md5_path = Path(config.md5_path)

print('chroma_db 是否存在:', persist_path.exists())
print('md5.txt 是否存在:', md5_path.exists())

if persist_path.exists():
    print('chroma_db 下的文件:')
    for item in sorted(persist_path.rglob('*')):
        if item.is_file():
            print('-', item.resolve())

if md5_path.exists():
    print('md5 记录条数:', len(md5_path.read_text(encoding='utf-8').splitlines()))

chroma_db 是否存在: True
md5.txt 是否存在: True
chroma_db 下的文件:
- D:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例\chroma_db\chroma.sqlite3
- D:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例\chroma_db\eeb78204-de20-46fd-8e31-e70ea9c6cd67\data_level0.bin
- D:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例\chroma_db\eeb78204-de20-46fd-8e31-e70ea9c6cd67\header.bin
- D:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例\chroma_db\eeb78204-de20-46fd-8e31-e70ea9c6cd67\length.bin
- D:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例\chroma_db\eeb78204-de20-46fd-8e31-e70ea9c6cd67\link_lists.bin
md5 记录条数: 3


## 09. 不问大模型，先单独测试“查资料”

RAG 分两步：先查资料，再回答。

这一节先只测“查资料”这一步，确认知识库检索是正常的。

In [13]:
# 创建 embedding 和 retriever
embedding = DashScopeEmbeddings(model=config.embedding_model_name)
retriever = VectorStoreService(embedding=embedding).get_retriever()

query = '我身高178cm，体重160斤，应该推荐什么尺码？'
docs = retriever.invoke(query)

print('问题:', query)
print('检索到的文档数量:', len(docs))

for idx, doc in enumerate(docs, start=1):
    print(f'--- 文档 {idx} ---')
    print(doc.page_content[:300])
    print('metadata =', doc.metadata)

问题: 我身高178cm，体重160斤，应该推荐什么尺码？
检索到的文档数量: 1
--- 文档 1 ---
身高：155-165cm， 体重：75-95 斤，建议尺码S。
身高：160-170cm， 体重：90-115斤，建议尺码M。
身高：165-175cm， 体重：115-135斤，建议尺码L。
身高：170-178cm， 体重：130-150斤，建议尺码XL。
身高：175-182cm， 体重：145-165斤，建议尺码2XL。
身高：178-185cm， 体重：160-180斤，建议尺码3XL。
身高：180-190cm， 体重：180-210斤，建议尺码4XL。
身高：190cm+，体重：210斤+，建议尺码5XL。
metadata = {'source': '尺码推荐.txt', 'create_time': '2026-01-18 16:05:35', 'operator': '小曹'}


## 10. 跑一次完整的 RAG 问答

现在才是真正的“先查资料，再回答”。

这里的 `RagService` 会自动把检索、拼 Prompt、调用大模型这几步连起来。

In [14]:
# 创建 RAG 服务对象
rag_service = RagService()
session_config = {'configurable': {'session_id': 'notebook_demo_user'}}

question = '针织毛衣如何保养？'
answer = rag_service.chain.invoke({'input': question}, session_config)

print('问题:', question)
print('回答:', answer)

System: 以我提供的已知参考资料为主，简洁和专业的回答用户问题。参考资料:文档片段：3. 加绒牛仔材质（冬季加绒牛仔裤、加绒牛仔外套）

洗涤：水温≤30℃，中性洗涤剂；翻面清洗，机洗选轻柔模式；避免长时间浸泡，减少绒层脱落。

养护：翻面阴干，避免暴晒；收纳时折叠平放，避免重压破坏绒层；清洗后及时晾干，防止发霉产生异味。

4. 保暖内衣材质（纯棉保暖内衣、德绒保暖内衣）

洗涤：可机洗或手洗，水温≤30℃，中性洗涤剂；机洗选轻柔模式，避免强力旋转；禁止使用漂白剂。

养护：阴凉阴干或阳光下短时间晾晒；收纳时折叠平整，放干燥处；德绒材质避免高温熨烫，防止破坏保暖纤维。

五、通用养护小贴士

1. 不同材质衣物分开清洗，避免染色、磨损；洗涤剂充分溶解后再放衣物，避免局部变色。

2. 污渍及时处理，时间越久越难去除；不同污渍针对性处理（油渍用洗洁精原液，血渍用冷水浸泡）。

3. 熨烫前查看衣物洗标，按材质调整温度；首次熨烫先在衣物内侧测试，避免烫伤。

4. 长期存放的衣物，收纳前务必完全干燥，定期检查是否发霉、虫蛀。
文档元数据：{'operator': '小曹', 'source': '洗涤养护.txt', 'create_time': '2026-01-18 16:05:39'}

。
System: 并且我提供用户的对话历史记录，如下：
Human: 请回答用户提问：针织毛衣如何保养？
问题: 针织毛衣如何保养？
回答: 参考资料中未直接提及针织毛衣的保养方法。但可参考通用养护小贴士及类似材质（如保暖内衣、加绒牛仔）的建议，提供以下保养指导：

1. **洗涤**：建议手洗或机洗轻柔模式，水温≤30℃，使用中性洗涤剂；避免长时间浸泡和强力搓揉，以防变形或起球。

2. **晾干**：平铺阴干，避免悬挂导致拉伸变形，切勿暴晒。

3. **收纳**：折叠存放，避免重压；长期存放前确保完全干燥，并置于干燥处，定期检查是否发霉或虫蛀。

4. **熨烫**：如需熨烫，先查看洗标，选择合适温度，并在内侧测试，避免烫伤面料。

5. **去污**：污渍应及时处理，油渍可用洗洁精原液局部处理，血渍用冷水浸泡。


## 11. 上下文记忆

这个案例不是只能一问一答，还会把聊天记录保存在本地，所以同一个 `session_id` 下，多轮提问是能接上文的。

In [15]:
# 用同一个 session_id 连续问 3 次
multi_turn_questions = [
    '我身高178cm，体重160斤，适合什么尺码？',
    '如果我是黄皮，要去面试，颜色上怎么选？',
    '那针织类衣服平时怎么洗比较稳妥？'
]

for idx, question in enumerate(multi_turn_questions, start=1):
    answer = rag_service.chain.invoke({'input': question}, session_config)
    print(f'第 {idx} 轮问题: {question}')
    print('回答:', answer)
    print('-' * 60)

System: 以我提供的已知参考资料为主，简洁和专业的回答用户问题。参考资料:文档片段：身高：155-165cm， 体重：75-95 斤，建议尺码S。
身高：160-170cm， 体重：90-115斤，建议尺码M。
身高：165-175cm， 体重：115-135斤，建议尺码L。
身高：170-178cm， 体重：130-150斤，建议尺码XL。
身高：175-182cm， 体重：145-165斤，建议尺码2XL。
身高：178-185cm， 体重：160-180斤，建议尺码3XL。
身高：180-190cm， 体重：180-210斤，建议尺码4XL。
身高：190cm+，体重：210斤+，建议尺码5XL。
文档元数据：{'operator': '小曹', 'source': '尺码推荐.txt', 'create_time': '2026-01-18 16:05:35'}

。
System: 并且我提供用户的对话历史记录，如下：
Human: 针织毛衣如何保养？
AI: 参考资料中未直接提及针织毛衣的保养方法。但可参考通用养护小贴士及类似材质（如保暖内衣、加绒牛仔）的建议，提供以下保养指导：

1. **洗涤**：建议手洗或机洗轻柔模式，水温≤30℃，使用中性洗涤剂；避免长时间浸泡和强力搓揉，以防变形或起球。

2. **晾干**：平铺阴干，避免悬挂导致拉伸变形，切勿暴晒。

3. **收纳**：折叠存放，避免重压；长期存放前确保完全干燥，并置于干燥处，定期检查是否发霉或虫蛀。

4. **熨烫**：如需熨烫，先查看洗标，选择合适温度，并在内侧测试，避免烫伤面料。

5. **去污**：污渍应及时处理，油渍可用洗洁精原液局部处理，血渍用冷水浸泡。
Human: 请回答用户提问：我身高178cm，体重160斤，适合什么尺码？
第 1 轮问题: 我身高178cm，体重160斤，适合什么尺码？
回答: 根据您提供的身高178cm、体重160斤，对照尺码推荐表：

- 身高178-185cm，体重160-180斤，建议尺码为 **3XL**。

因此，您适合选择 **3XL** 尺码。
------------------------------------------------------------
System: 以我提供的已知参考资料为主，简洁和专业的回答用

## 12. 聊天记录保存

如果想知道“为什么能记住我刚才问过什么”，就看这里。

聊天历史会被写进 `chat_history` 目录。

In [16]:
history_file = Path('chat_history') / 'notebook_demo_user'
print('聊天历史文件路径:', history_file)

if history_file.exists():
    history_data = json.loads(history_file.read_text(encoding='utf-8'))
    print('消息条数:', len(history_data))
    print('前两条消息示例:')
    print(json.dumps(history_data[:2], ensure_ascii=False, indent=2))
else:
    print('还没有历史文件。请先运行上一节的多轮对话。')

聊天历史文件路径: chat_history\notebook_demo_user
消息条数: 8
前两条消息示例:
[
  {
    "type": "human",
    "data": {
      "content": "针织毛衣如何保养？",
      "additional_kwargs": {},
      "response_metadata": {},
      "type": "human",
      "name": null,
      "id": null
    }
  },
  {
    "type": "ai",
    "data": {
      "content": "参考资料中未直接提及针织毛衣的保养方法。但可参考通用养护小贴士及类似材质（如保暖内衣、加绒牛仔）的建议，提供以下保养指导：\n\n1. **洗涤**：建议手洗或机洗轻柔模式，水温≤30℃，使用中性洗涤剂；避免长时间浸泡和强力搓揉，以防变形或起球。\n\n2. **晾干**：平铺阴干，避免悬挂导致拉伸变形，切勿暴晒。\n\n3. **收纳**：折叠存放，避免重压；长期存放前确保完全干燥，并置于干燥处，定期检查是否发霉或虫蛀。\n\n4. **熨烫**：如需熨烫，先查看洗标，选择合适温度，并在内侧测试，避免烫伤面料。\n\n5. **去污**：污渍应及时处理，油渍可用洗洁精原液局部处理，血渍用冷水浸泡。",
      "additional_kwargs": {},
      "response_metadata": {},
      "type": "ai",
      "name": null,
      "id": null,
      "tool_calls": [],
      "invalid_tool_calls": [],
      "usage_metadata": null
    }
  }
]


## 13. 每个 Python 文件分别是干什么的

这是最适合小白建立全局认识的一节。你不用一开始就读源码，先记住“谁负责什么”就够了。

In [17]:
module_map = {
    'config_data.py': '放模型名、路径、切分参数等配置',
    'knowledge_base.py': '把文本切分后写入向量库',
    'vector_stores.py': '把向量库包装成检索器 retriever',
    'rag.py': '把检索 + Prompt + 大模型串起来',
    'file_history_store.py': '把聊天记录保存到本地文件',
    'app_file_uploader.py': '提供上传 txt 文件的网页入口',
    'app_qa.py': '提供智能问答网页入口',
}

for name, desc in module_map.items():
    print(f'{name}: {desc}')

config_data.py: 放模型名、路径、切分参数等配置
knowledge_base.py: 把文本切分后写入向量库
vector_stores.py: 把向量库包装成检索器 retriever
rag.py: 把检索 + Prompt + 大模型串起来
file_history_store.py: 把聊天记录保存到本地文件
app_file_uploader.py: 提供上传 txt 文件的网页入口
app_qa.py: 提供智能问答网页入口


## 14. 如果不用 Notebook，怎么直接跑网页

这个项目有两个 Streamlit 页面。

一个页面负责上传资料，一个页面负责提问聊天。可以在终端里直接运行它们：

先切换到当前路径（路径可以直接去文件夹或vscode中复制，也可以在最开始的01单元格查看）

我的路径是

cd d:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例

powershell

streamlit run .\app_file_uploader.py

每次运行切都要切到代码存在的目录

powershell

streamlit run .\app_qa.py